# Разбивка на одноклассовые классификации (LB - 0.818)
Метрика задачи позволяет переформулировать задачу из multi-label классификации и получить более мелкие задачи классификации, вплоть до 41 задачи одноклассовой классификации. Задачи одноклассовой классификации можно решать независимо, разделив их между участниками команды и/или между компьютерами.

In [22]:
conda create -n pycaret_env python=3.10 -y

Retrieving notices: done
Channels:
 - defaults
Platform: osx-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 25.7.0

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /opt/anaconda3/envs/pycaret_env

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    openssl-3.0.17             |       hee2dfae_0         4.6 MB
    pip-26.0.1                 |     pyhc872135_0         1.1 MB
    python-3.10.18             |       hc958d9f_0        13.1 MB
    setuptools-78.1.1          |  py310hecd8cb5_0         1.6 MB
    tzdata-2025c               |       he532380_0         118 KB
    wheel-0.45.1               |  py310hecd8cb5_0         119 KB
    ------------------------------------------------------------
 

In [23]:
conda activate pycaret_env


CondaError: Run 'conda init' before 'conda activate'


Note: you may need to restart the kernel to use updated packages.


In [24]:
pip install pycaret

Note: you may need to restart the kernel to use updated packages.


In [25]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

from catboost import Pool, CatBoostClassifier
from sklearn.metrics import roc_auc_score

In [26]:
data_path = 'data/'

In [27]:
%%time
#train = pl.read_parquet('data/train_main_features.parquet')
#test = pl.read_parquet('data/test_main_features.parquet')

train = pl.read_parquet(data_path+'train/'+'train_main_features.parquet'
                       ).fill_null(-99)
test = pl.read_parquet(data_path+'test/'+'test_main_features.parquet'
                      ).fill_null(-99)

print('Тренировочные данные:', train.shape)
print('Тестовые данные:', test.shape)
train

Тренировочные данные: (750000, 200)
Тестовые данные: (250000, 200)
CPU times: user 671 ms, sys: 2.07 s, total: 2.74 s
Wall time: 724 ms


customer_id,cat_feature_1,cat_feature_2,cat_feature_3,cat_feature_4,cat_feature_5,cat_feature_6,cat_feature_7,cat_feature_8,cat_feature_9,cat_feature_10,cat_feature_11,cat_feature_12,cat_feature_13,cat_feature_14,cat_feature_15,cat_feature_16,cat_feature_17,cat_feature_18,cat_feature_19,cat_feature_20,cat_feature_21,cat_feature_22,cat_feature_23,cat_feature_24,cat_feature_25,cat_feature_26,cat_feature_27,cat_feature_28,cat_feature_29,cat_feature_30,cat_feature_31,cat_feature_32,cat_feature_33,cat_feature_34,cat_feature_35,cat_feature_36,…,num_feature_96,num_feature_97,num_feature_98,num_feature_99,num_feature_100,num_feature_101,num_feature_102,num_feature_103,num_feature_104,num_feature_105,num_feature_106,num_feature_107,num_feature_108,num_feature_109,num_feature_110,num_feature_111,num_feature_112,num_feature_113,num_feature_114,num_feature_115,num_feature_116,num_feature_117,num_feature_118,num_feature_119,num_feature_120,num_feature_121,num_feature_122,num_feature_123,num_feature_124,num_feature_125,num_feature_126,num_feature_127,num_feature_128,num_feature_129,num_feature_130,num_feature_131,num_feature_132
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1000001,1.0,0.0,2.0,1.0,2.0,3.0,2.0,2.0,4.0,2.0,0.0,0.0,0.0,1.0,2.0,0.0,2.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,1.0,2.0,1.0,2.0,2.0,0.0,0.0,212.0,0.0,0.0,…,-0.284519,-99.0,-0.004499,-99.0,-0.050159,-0.002297,-0.043592,-0.061488,0.450342,-99.0,0.0,-99.0,-0.060492,-0.279105,-99.0,-0.429813,-99.0,-0.009654,-0.293036,-99.0,-0.493959,-0.019079,-99.0,-99.0,-99.0,-99.0,-0.001357,-0.031281,-0.046146,-99.0,-0.445279,-99.0,-99.0,-0.107666,-0.418616,-99.0,-99.0
1000002,1.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,212.0,1.0,0.0,…,-0.284519,0.460275,-0.004278,-99.0,-0.556244,-0.002297,-0.043592,-0.061488,1.998252,-0.009552,0.0,-0.005762,-0.060492,-0.279105,-99.0,-0.429813,-99.0,-0.009654,-0.293036,-0.004421,-0.256445,-0.014154,-99.0,-0.24167,-99.0,-99.0,-0.001357,-0.031281,-0.046146,-0.10217,1.550722,-99.0,-99.0,-0.170724,-0.805771,-0.397803,-0.373734
1000003,1.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,212.0,0.0,0.0,…,-0.284519,-0.264397,-0.004278,-99.0,-99.0,-0.002297,-0.043592,-0.061488,-0.264078,-99.0,0.0,-0.26649,-0.060492,-0.279105,-99.0,-0.429813,-99.0,-0.009654,-0.293036,-0.004421,-0.57313,-0.019124,-99.0,-0.24167,-99.0,-99.0,-0.001357,-0.031281,-0.046146,-99.0,-0.475778,-99.0,-99.0,-0.170724,-0.602005,-0.397803,-0.373734
1000004,1.0,0.0,2.0,1.0,2.0,3.0,2.0,2.0,3.0,2.0,0.0,0.0,0.0,1.0,2.0,2.0,2.0,2.0,2.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,2.0,1.0,2.0,2.0,0.0,0.0,212.0,0.0,0.0,…,-0.284519,-99.0,-0.004499,-99.0,-99.0,-0.002297,-0.043592,-0.061488,0.688482,-0.009552,0.0,-99.0,-0.060492,-99.0,-99.0,-0.429813,-99.0,-0.009654,-0.293036,-99.0,-0.57313,-99.0,-99.0,-0.505441,-99.0,0.714631,-0.001357,-0.031281,-0.046146,-99.0,-0.475778,0.111196,0.116695,-99.0,-0.724265,-99.0,-99.0
1000005,1.0,2.0,0.0,1.0,0.0,3.0,0.0,0.0,2.0,1.0,2.0,2.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,0.0,2.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,2.0,2.0,212.0,2.0,2.0,…,-0.284519,-0.264397,-0.004499,-99.0,-99.0,-0.002297,-0.043592,-99.0,-0.264078,-99.0,0.0,-0.26649,-99.0,-0.279105,-99.0,-0.429813,-99.0,-0.009654,-0.293036,-0.004421,-0.57313,-0.018674,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-0.046146,-99.0,-99.0,-99.0,-99.0,-0.107666,-99.0,-0.397803,-0.373734
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1749996,1.0,0.0,2.0,0.0,2.0,3.0,2.0,2.0,4.0,2.0,0.0,0.0,0.0,1.0,2.0,0.0,2.0,2.0,2.0,0.0,0.0,2.0,2.0,0.0,0.0

In [28]:
missing_values = (
    train.null_count()
    .unpivot(variable_name="Column", value_name="Missing Values")
    .filter(pl.col("Missing Values") > 0)
    .sort("Missing Values", descending=True)
)

print('\nКоличество пропусков в признаках:\n', missing_values)


Количество пропусков в признаках:
 shape: (0, 2)
┌────────┬────────────────┐
│ Column ┆ Missing Values │
│ ---    ┆ ---            │
│ str    ┆ u32            │
╞════════╪════════════════╡
└────────┴────────────────┘


In [29]:
cat_feature_names = [
    col_name for col_name in train.columns 
    if col_name.startswith("cat_feature")
]
cat_feature_names

['cat_feature_1',
 'cat_feature_2',
 'cat_feature_3',
 'cat_feature_4',
 'cat_feature_5',
 'cat_feature_6',
 'cat_feature_7',
 'cat_feature_8',
 'cat_feature_9',
 'cat_feature_10',
 'cat_feature_11',
 'cat_feature_12',
 'cat_feature_13',
 'cat_feature_14',
 'cat_feature_15',
 'cat_feature_16',
 'cat_feature_17',
 'cat_feature_18',
 'cat_feature_19',
 'cat_feature_20',
 'cat_feature_21',
 'cat_feature_22',
 'cat_feature_23',
 'cat_feature_24',
 'cat_feature_25',
 'cat_feature_26',
 'cat_feature_27',
 'cat_feature_28',
 'cat_feature_29',
 'cat_feature_30',
 'cat_feature_31',
 'cat_feature_32',
 'cat_feature_33',
 'cat_feature_34',
 'cat_feature_35',
 'cat_feature_36',
 'cat_feature_37',
 'cat_feature_38',
 'cat_feature_39',
 'cat_feature_40',
 'cat_feature_41',
 'cat_feature_42',
 'cat_feature_43',
 'cat_feature_44',
 'cat_feature_45',
 'cat_feature_46',
 'cat_feature_47',
 'cat_feature_48',
 'cat_feature_49',
 'cat_feature_50',
 'cat_feature_51',
 'cat_feature_52',
 'cat_feature_53',
 '

In [30]:
train = train.with_columns(
    pl.col(cat_feature_names).cast(pl.Int32)
)
train 

customer_id,cat_feature_1,cat_feature_2,cat_feature_3,cat_feature_4,cat_feature_5,cat_feature_6,cat_feature_7,cat_feature_8,cat_feature_9,cat_feature_10,cat_feature_11,cat_feature_12,cat_feature_13,cat_feature_14,cat_feature_15,cat_feature_16,cat_feature_17,cat_feature_18,cat_feature_19,cat_feature_20,cat_feature_21,cat_feature_22,cat_feature_23,cat_feature_24,cat_feature_25,cat_feature_26,cat_feature_27,cat_feature_28,cat_feature_29,cat_feature_30,cat_feature_31,cat_feature_32,cat_feature_33,cat_feature_34,cat_feature_35,cat_feature_36,…,num_feature_96,num_feature_97,num_feature_98,num_feature_99,num_feature_100,num_feature_101,num_feature_102,num_feature_103,num_feature_104,num_feature_105,num_feature_106,num_feature_107,num_feature_108,num_feature_109,num_feature_110,num_feature_111,num_feature_112,num_feature_113,num_feature_114,num_feature_115,num_feature_116,num_feature_117,num_feature_118,num_feature_119,num_feature_120,num_feature_121,num_feature_122,num_feature_123,num_feature_124,num_feature_125,num_feature_126,num_feature_127,num_feature_128,num_feature_129,num_feature_130,num_feature_131,num_feature_132
i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1000001,1,0,2,1,2,3,2,2,4,2,0,0,0,1,2,0,2,2,2,0,0,0,2,0,0,0,1,2,1,2,2,0,0,212,0,0,…,-0.284519,-99.0,-0.004499,-99.0,-0.050159,-0.002297,-0.043592,-0.061488,0.450342,-99.0,0.0,-99.0,-0.060492,-0.279105,-99.0,-0.429813,-99.0,-0.009654,-0.293036,-99.0,-0.493959,-0.019079,-99.0,-99.0,-99.0,-99.0,-0.001357,-0.031281,-0.046146,-99.0,-0.445279,-99.0,-99.0,-0.107666,-0.418616,-99.0,-99.0
1000002,1,0,0,1,0,3,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,212,1,0,…,-0.284519,0.460275,-0.004278,-99.0,-0.556244,-0.002297,-0.043592,-0.061488,1.998252,-0.009552,0.0,-0.005762,-0.060492,-0.279105,-99.0,-0.429813,-99.0,-0.009654,-0.293036,-0.004421,-0.256445,-0.014154,-99.0,-0.24167,-99.0,-99.0,-0.001357,-0.031281,-0.046146,-0.10217,1.550722,-99.0,-99.0,-0.170724,-0.805771,-0.397803,-0.373734
1000003,1,0,0,1,0,3,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,1,0,0,0,0,0,0,212,0,0,…,-0.284519,-0.264397,-0.004278,-99.0,-99.0,-0.002297,-0.043592,-0.061488,-0.264078,-99.0,0.0,-0.26649,-0.060492,-0.279105,-99.0,-0.429813,-99.0,-0.009654,-0.293036,-0.004421,-0.57313,-0.019124,-99.0,-0.24167,-99.0,-99.0,-0.001357,-0.031281,-0.046146,-99.0,-0.475778,-99.0,-99.0,-0.170724,-0.602005,-0.397803,-0.373734
1000004,1,0,2,1,2,3,2,2,3,2,0,0,0,1,2,2,2,2,2,0,0,2,2,0,0,0,0,2,1,2,2,0,0,212,0,0,…,-0.284519,-99.0,-0.004499,-99.0,-99.0,-0.002297,-0.043592,-0.061488,0.688482,-0.009552,0.0,-99.0,-0.060492,-99.0,-99.0,-0.429813,-99.0,-0.009654,-0.293036,-99.0,-0.57313,-99.0,-99.0,-0.505441,-99.0,0.714631,-0.001357,-0.031281,-0.046146,-99.0,-0.475778,0.111196,0.116695,-99.0,-0.724265,-99.0,-99.0
1000005,1,2,0,1,0,3,0,0,2,1,2,2,2,0,1,0,0,0,0,2,2,2,0,2,2,2,2,0,0,0,0,2,2,212,2,2,…,-0.284519,-0.264397,-0.004499,-99.0,-99.0,-0.002297,-0.043592,-99.0,-0.264078,-99.0,0.0,-0.26649,-99.0,-0.279105,-99.0,-0.429813,-99.0,-0.009654,-0.293036,-0.004421,-0.57313,-0.018674,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-0.046146,-99.0,-99.0,-99.0,-99.0,-0.107666,-99.0,-0.397803,-0.373734
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1749996,1,0,2,0,2,3,2,2,4,2,0,0,0,1,2,0,2,2,2,0,0,2,2,0,0,0,0,2,1,2,2,0,0,212,0,0,…,-99.0,-99.0,-0.004499,-0.150243,-99.0,-99.0,-99.0,2.831928,-99.0,-99.0,-99.0,-99.0,-0.060492,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-0.019236,-99.0,-0.505441,-99.0,-99.0,-0.001357,-0.031281,-99.0,-99.0,-0.475778,-99.0,-99.0,-99.0,0.335315,-99.0,-99.0
1749997,1,0,2,1,2,3,2,2,2,2,0,0,0,1,2,0,2,2,2,0,0,0,2,0,0,0,0,2,1,2,2,0,0,212,1,0,…,-0.284519,-99.0,-0.004499,-99.0,-0.5100

In [31]:
target = pl.read_parquet(data_path+'baseline/'+'train_target.parquet')
target
#shape: (750_000, 42)

customer_id,target_1_1,target_1_2,target_1_3,target_1_4,target_1_5,target_2_1,target_2_2,target_2_3,target_2_4,target_2_5,target_2_6,target_2_7,target_2_8,target_3_1,target_3_2,target_3_3,target_3_4,target_3_5,target_4_1,target_5_1,target_5_2,target_6_1,target_6_2,target_6_3,target_6_4,target_6_5,target_7_1,target_7_2,target_7_3,target_8_1,target_8_2,target_8_3,target_9_1,target_9_2,target_9_3,target_9_4,target_9_5,target_9_6,target_9_7,target_9_8,target_10_1
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1000003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1000004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1000005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1749996,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1749997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1749998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [32]:
target_columns = [col for col in target.columns if col.startswith("target")]
target_columns

['target_1_1',
 'target_1_2',
 'target_1_3',
 'target_1_4',
 'target_1_5',
 'target_2_1',
 'target_2_2',
 'target_2_3',
 'target_2_4',
 'target_2_5',
 'target_2_6',
 'target_2_7',
 'target_2_8',
 'target_3_1',
 'target_3_2',
 'target_3_3',
 'target_3_4',
 'target_3_5',
 'target_4_1',
 'target_5_1',
 'target_5_2',
 'target_6_1',
 'target_6_2',
 'target_6_3',
 'target_6_4',
 'target_6_5',
 'target_7_1',
 'target_7_2',
 'target_7_3',
 'target_8_1',
 'target_8_2',
 'target_8_3',
 'target_9_1',
 'target_9_2',
 'target_9_3',
 'target_9_4',
 'target_9_5',
 'target_9_6',
 'target_9_7',
 'target_9_8',
 'target_10_1']

## Обучение модели
Обучение 41 независимой модели бинарной классификации

In [36]:
model = CatBoostClassifier(
    #one_hot_max_size=1024*2,
    one_hot_max_size=256,
    
    loss_function = 'Logloss', 
    custom_metric=["Accuracy","AUC"],
    eval_metric="AUC",
    nan_mode = 'Min', 
    random_seed = 1234,

    # task_type="GPU",
    task_type="CPU",   # На Mac работает только CPU
    verbose = 100
)

In [37]:
%%time
# limit=500_000
limit=700_000
for tcols in target_columns:
    print(tcols)
    train_pool = Pool(data = train[:limit].drop("customer_id").to_pandas(), 
                      label = (
                          target[:limit]
                              .select(tcols)
                              # .drop("customer_id")
                              .to_pandas(), 
                          # cat_features = cat_feature_names
                      )
                     )

    valid_pool = Pool(data = train[limit:].drop("customer_id").to_pandas(), 
                      label = (
                          target[limit:]
                              .select(tcols)
                              # .drop("customer_id")
                              .to_pandas(),
                          # cat_features = cat_feature_names
                      )
                     )
    
    model.fit(train_pool,
              plot=True,
              eval_set=valid_pool,
             )
    model.save_model(f"wocat{tcols}.cbm")

target_1_1


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Learning rate set to 0.159826
0:	test: 0.5834810	best: 0.5834810 (0)	total: 167ms	remaining: 2m 46s
100:	test: 0.9067562	best: 0.9067912 (97)	total: 7.32s	remaining: 1m 5s
200:	test: 0.9116913	best: 0.9117064 (199)	total: 13.5s	remaining: 53.5s
300:	test: 0.9145190	best: 0.9145190 (300)	total: 19.3s	remaining: 44.8s
400:	test: 0.9157701	best: 0.9157711 (399)	total: 25s	remaining: 37.4s
500:	test: 0.9167966	best: 0.9168130 (492)	total: 30.8s	remaining: 30.7s
600:	test: 0.9167192	best: 0.9169169 (591)	total: 36.5s	remaining: 24.2s
700:	test: 0.9170815	best: 0.9170910 (697)	total: 42.1s	remaining: 18s
800:	test: 0.9171724	best: 0.9172335 (722)	total: 47.6s	remaining: 11.8s
900:	test: 0.9169574	best: 0.9173695 (808)	total: 53.4s	remaining: 5.87s
999:	test: 0.9169469	best: 0.9173695 (808)	total: 58.9s	remaining: 0us

bestTest = 0.9173694636
bestIteration = 808

Shrink model to first 809 iterations.
target_1_2


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Learning rate set to 0.159826
0:	test: 0.6331434	best: 0.6331434 (0)	total: 90.6ms	remaining: 1m 30s
100:	test: 0.8029972	best: 0.8034136 (96)	total: 6.19s	remaining: 55.1s
200:	test: 0.8066138	best: 0.8067777 (172)	total: 11.8s	remaining: 46.9s
300:	test: 0.8054688	best: 0.8072139 (205)	total: 17.5s	remaining: 40.7s
400:	test: 0.8040988	best: 0.8072139 (205)	total: 23.1s	remaining: 34.6s
500:	test: 0.8036074	best: 0.8072139 (205)	total: 28.8s	remaining: 28.7s
600:	test: 0.8044930	best: 0.8072139 (205)	total: 34.8s	remaining: 23.1s
700:	test: 0.8048938	best: 0.8072139 (205)	total: 40.7s	remaining: 17.4s
800:	test: 0.8034653	best: 0.8072139 (205)	total: 46.6s	remaining: 11.6s
900:	test: 0.8002411	best: 0.8072139 (205)	total: 52.4s	remaining: 5.76s
999:	test: 0.8008910	best: 0.8072139 (205)	total: 58s	remaining: 0us

bestTest = 0.8072138771
bestIteration = 205

Shrink model to first 206 iterations.
target_1_3


...

Learning rate set to 0.159826
0:	test: 0.6611838	best: 0.6611838 (0)	total: 108ms	remaining: 1m 48s
100:	test: 0.7461177	best: 0.7461177 (100)	total: 7.68s	remaining: 1m 8s
200:	test: 0.7485284	best: 0.7486160 (195)	total: 14.5s	remaining: 57.8s
300:	test: 0.7494334	best: 0.7494780 (293)	total: 21.2s	remaining: 49.3s
400:	test: 0.7502272	best: 0.7502424 (398)	total: 28.1s	remaining: 41.9s
500:	test: 0.7500271	best: 0.7502424 (398)	total: 34.6s	remaining: 34.4s
600:	test: 0.7498211	best: 0.7502548 (525)	total: 41.8s	remaining: 27.8s
700:	test: 0.7494979	best: 0.7502548 (525)	total: 48.6s	remaining: 20.7s
800:	test: 0.7489932	best: 0.7502548 (525)	total: 54.8s	remaining: 13.6s
900:	test: 0.7487515	best: 0.7502548 (525)	total: 1m 1s	remaining: 6.72s
999:	test: 0.7489720	best: 0.7502548 (525)	total: 1m 7s	remaining: 0us

bestTest = 0.7502547977
bestIteration = 525

Shrink model to first 526 iterations.
target_9_8


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Learning rate set to 0.159826
0:	test: 0.5704905	best: 0.5704905 (0)	total: 88.7ms	remaining: 1m 28s
100:	test: 0.9172480	best: 0.9172480 (100)	total: 7.04s	remaining: 1m 2s
200:	test: 0.9198825	best: 0.9198866 (197)	total: 12.9s	remaining: 51.2s
300:	test: 0.9209638	best: 0.9210579 (291)	total: 18.7s	remaining: 43.4s
400:	test: 0.9220060	best: 0.9220119 (394)	total: 24.6s	remaining: 36.7s
500:	test: 0.9225488	best: 0.9225612 (499)	total: 31.7s	remaining: 31.5s
600:	test: 0.9228431	best: 0.9228996 (551)	total: 38.7s	remaining: 25.7s
700:	test: 0.9232541	best: 0.9233435 (696)	total: 45.2s	remaining: 19.3s
800:	test: 0.9240014	best: 0.9240689 (787)	total: 51.6s	remaining: 12.8s
900:	test: 0.9241177	best: 0.9242638 (867)	total: 57.7s	remaining: 6.34s
999:	test: 0.9246701	best: 0.9246917 (989)	total: 1m 3s	remaining: 0us

bestTest = 0.9246917125
bestIteration = 989

Shrink model to first 990 iterations.
target_10_1


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Learning rate set to 0.159826
0:	test: 0.7031518	best: 0.7031518 (0)	total: 113ms	remaining: 1m 53s
100:	test: 0.7394039	best: 0.7394039 (100)	total: 7.73s	remaining: 1m 8s
200:	test: 0.7434267	best: 0.7434428 (198)	total: 15.2s	remaining: 1m
300:	test: 0.7449829	best: 0.7449829 (300)	total: 22.2s	remaining: 51.6s
400:	test: 0.7455434	best: 0.7455636 (387)	total: 29.2s	remaining: 43.7s
500:	test: 0.7462957	best: 0.7463095 (499)	total: 36.2s	remaining: 36.1s
600:	test: 0.7463355	best: 0.7463513 (566)	total: 43.1s	remaining: 28.6s
700:	test: 0.7465923	best: 0.7465924 (695)	total: 50.4s	remaining: 21.5s
800:	test: 0.7468680	best: 0.7469202 (776)	total: 57.2s	remaining: 14.2s
900:	test: 0.7470894	best: 0.7471569 (864)	total: 1m 3s	remaining: 6.99s
999:	test: 0.7470772	best: 0.7471569 (864)	total: 1m 9s	remaining: 0us

bestTest = 0.747156876
bestIteration = 864

Shrink model to first 865 iterations.
CPU times: user 6h 37min 54s, sys: 43min 34s, total: 7h 21min 29s
Wall time: 41min 22s


**Windows:**
- CPU times: user 14min 14s, sys: 6min 44s, total: 20min 59s
- Wall time: 9min 41s

**Mac:**
- CPU times: user 6h 37min 54s, sys: 43min 34s, total: 7h 21min 29s
- Wall time: 41min 22s

In [39]:
test = pl.read_parquet(data_path+'test/'+'test_main_features.parquet'
                      ).fill_null(-99)

test = test.with_columns(
    pl.col(cat_feature_names).cast(pl.Int32)
)
test

customer_id,cat_feature_1,cat_feature_2,cat_feature_3,cat_feature_4,cat_feature_5,cat_feature_6,cat_feature_7,cat_feature_8,cat_feature_9,cat_feature_10,cat_feature_11,cat_feature_12,cat_feature_13,cat_feature_14,cat_feature_15,cat_feature_16,cat_feature_17,cat_feature_18,cat_feature_19,cat_feature_20,cat_feature_21,cat_feature_22,cat_feature_23,cat_feature_24,cat_feature_25,cat_feature_26,cat_feature_27,cat_feature_28,cat_feature_29,cat_feature_30,cat_feature_31,cat_feature_32,cat_feature_33,cat_feature_34,cat_feature_35,cat_feature_36,…,num_feature_96,num_feature_97,num_feature_98,num_feature_99,num_feature_100,num_feature_101,num_feature_102,num_feature_103,num_feature_104,num_feature_105,num_feature_106,num_feature_107,num_feature_108,num_feature_109,num_feature_110,num_feature_111,num_feature_112,num_feature_113,num_feature_114,num_feature_115,num_feature_116,num_feature_117,num_feature_118,num_feature_119,num_feature_120,num_feature_121,num_feature_122,num_feature_123,num_feature_124,num_feature_125,num_feature_126,num_feature_127,num_feature_128,num_feature_129,num_feature_130,num_feature_131,num_feature_132
i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1750001,1,0,2,0,2,3,2,2,4,0,0,0,0,1,0,0,2,2,2,0,0,0,2,0,0,0,1,2,1,2,2,0,0,212,0,0,…,0.503368,-0.264397,-0.004499,-0.150243,-0.241966,-0.002297,-0.043592,-0.061488,-0.383148,-0.009552,0.0,-0.26649,-0.060492,-0.279105,-99.0,-0.429813,-99.0,-0.009654,-0.293036,-99.0,-0.493959,-0.01556,-99.0,-99.0,-99.0,-99.0,-0.001357,-0.031281,-0.046146,0.062409,-0.475778,-99.0,-99.0,0.018451,-0.765018,-0.397803,-0.373734
1750002,0,0,2,0,2,3,2,2,4,0,0,0,0,1,0,0,2,2,2,1,0,0,2,1,0,0,1,2,1,2,2,1,0,212,1,0,…,-0.284519,-0.264397,-0.004278,0.386748,-0.620003,-0.002297,-0.043592,0.367167,-0.025938,-99.0,0.0,-0.26649,-0.060492,-0.279105,-99.0,-0.429813,-99.0,-0.009654,3.569151,-99.0,-0.57313,-0.015727,-99.0,-99.0,-0.3427,-99.0,-0.001357,-0.031281,-0.046146,-99.0,-0.475778,-99.0,-99.0,-0.170724,-0.357487,-0.397803,-0.373734
1750003,1,0,2,1,2,3,2,2,4,2,0,0,0,1,2,0,2,2,2,0,0,0,2,0,0,0,1,2,1,2,2,1,0,212,0,0,…,-99.0,-0.264397,-0.004499,-99.0,-99.0,-0.002297,-99.0,-0.061488,-0.264078,-0.009468,-99.0,-0.26649,-0.060492,-0.279105,-99.0,-99.0,-99.0,-0.009654,-99.0,-99.0,-99.0,-0.01934,-99.0,0.022101,-99.0,-99.0,-99.0,-0.031281,-99.0,-99.0,-0.455446,-99.0,-99.0,0.018451,-0.785394,-99.0,-99.0
1750004,1,0,2,0,2,3,2,2,4,2,0,0,0,1,2,0,2,2,2,0,0,2,2,0,0,0,1,2,1,2,2,0,0,212,0,0,…,-0.284519,-99.0,-0.004499,-0.150243,-99.0,-0.002297,-0.043592,-0.061488,-0.383148,-99.0,0.0,-99.0,-0.060492,-99.0,-99.0,-0.429813,-99.0,-0.009654,-0.293036,-99.0,-0.018932,-0.015257,-99.0,-99.0,-0.388464,-99.0,-0.001357,-0.031281,-0.046146,0.054572,-0.475778,-99.0,-99.0,-99.0,-0.520499,-99.0,-99.0
1750005,0,1,2,0,2,3,2,2,4,2,0,0,0,1,2,0,2,2,2,1,0,0,2,1,1,0,1,2,1,2,2,1,0,212,0,0,…,-0.284519,-0.264397,-0.004499,-0.150243,1.663358,-0.002297,-0.043592,-0.061488,-0.383148,-99.0,0.0,-0.26649,-0.060492,-0.279105,-99.0,-0.429813,-99.0,-0.009654,-0.293036,-99.0,-0.57313,-0.017804,-99.0,-0.24167,-99.0,-0.138239,-0.001357,-0.031281,-0.046146,-99.0,1.533778,1.318607,-99.0,-0.170724,0.314939,-99.0,-99.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1999996,1,2,2,1,2,3,2,2,4,2,2,2,2,1,2,0,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,2,212,2,2,…,-99.0,-99.0,-0.004499,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0
1999997,1,2,0,1,0,3,0,0,4,1,2,2,2,0,0,0,0,0,0,2,2,0,0,2,2,2,2,0,0,0,0,2,2,212,2,2,…,-0.284519,-0.264397,-0.004499,-99.0,-0.761335,-0.002297,-0.043592,-

## Предсказание модели

In [40]:
test_pool = Pool(data = test.drop("customer_id").to_pandas(), 
                 #cat_features = cat_feature_names
                )
test_pool

Заметим, что колонки `target` почему-то разделены на группы:
|       | target_1 | taget_2 | target_3 | target_4 | target_5 | target_6 | taget_7 | taget_8 | taget_9 | target_10 |
|-------|----------|---------|----------|----------|----------|----------|---------|---------|---------|-----------|
| count | 5        | 8       | 5        | 1        | 2        | 5        | 3       | 3       | 8       | 1         |


In [41]:
for tcols in target_columns:
    print(tcols)
    model.load_model(f"wocat{tcols}.cbm")
    test_predict = model.predict(test_pool, prediction_type = "RawFormulaVal")
    test = test.with_columns(temp=test_predict).rename({"temp":tcols})
test

target_1_1
target_1_2
target_1_3
target_1_4
target_1_5
target_2_1
target_2_2
target_2_3
target_2_4
target_2_5
target_2_6
target_2_7
target_2_8
target_3_1
target_3_2
target_3_3
target_3_4
target_3_5
target_4_1
target_5_1
target_5_2
target_6_1
target_6_2
target_6_3
target_6_4
target_6_5
target_7_1
target_7_2
target_7_3
target_8_1
target_8_2
target_8_3
target_9_1
target_9_2
target_9_3
target_9_4
target_9_5
target_9_6
target_9_7
target_9_8
target_10_1


customer_id,cat_feature_1,cat_feature_2,cat_feature_3,cat_feature_4,cat_feature_5,cat_feature_6,cat_feature_7,cat_feature_8,cat_feature_9,cat_feature_10,cat_feature_11,cat_feature_12,cat_feature_13,cat_feature_14,cat_feature_15,cat_feature_16,cat_feature_17,cat_feature_18,cat_feature_19,cat_feature_20,cat_feature_21,cat_feature_22,cat_feature_23,cat_feature_24,cat_feature_25,cat_feature_26,cat_feature_27,cat_feature_28,cat_feature_29,cat_feature_30,cat_feature_31,cat_feature_32,cat_feature_33,cat_feature_34,cat_feature_35,cat_feature_36,…,target_1_5,target_2_1,target_2_2,target_2_3,target_2_4,target_2_5,target_2_6,target_2_7,target_2_8,target_3_1,target_3_2,target_3_3,target_3_4,target_3_5,target_4_1,target_5_1,target_5_2,target_6_1,target_6_2,target_6_3,target_6_4,target_6_5,target_7_1,target_7_2,target_7_3,target_8_1,target_8_2,target_8_3,target_9_1,target_9_2,target_9_3,target_9_4,target_9_5,target_9_6,target_9_7,target_9_8,target_10_1
i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1750001,1,0,2,0,2,3,2,2,4,0,0,0,0,1,0,0,2,2,2,0,0,0,2,0,0,0,1,2,1,2,2,0,0,212,0,0,…,-7.029726,-4.984127,-5.272237,-7.648119,-4.768143,-6.958062,-6.138118,-8.929741,-12.500473,-2.240438,-4.582098,-7.585328,-8.788304,-11.427572,-4.502799,-5.568636,-6.659411,-6.101653,-6.153649,-5.613236,-5.934041,-9.173555,-2.753256,-4.745554,-5.110236,-6.248015,-4.634518,-3.979078,-5.550732,-2.905332,-4.018138,-10.36281,-7.443879,-0.295655,-2.368205,-7.109498,-0.59118
1750002,0,0,2,0,2,3,2,2,4,0,0,0,0,1,0,0,2,2,2,1,0,0,2,1,0,0,1,2,1,2,2,1,0,212,1,0,…,-6.610275,-3.243938,-5.277364,-7.327784,-4.304502,-5.916581,-6.39367,-8.072554,-12.004719,-2.78021,-5.05643,-7.289385,-8.92129,-11.614137,-6.261377,-5.052025,-6.432152,-6.026409,-5.267215,-5.941416,-5.997136,-9.000745,-3.318007,-4.060681,-6.220353,-4.872623,-2.352105,-5.616156,-6.227298,-1.540115,-3.646324,-4.635172,-4.257187,-0.754386,-1.90217,-9.887006,-1.284431
1750003,1,0,2,1,2,3,2,2,4,2,0,0,0,1,2,0,2,2,2,0,0,0,2,0,0,0,1,2,1,2,2,1,0,212,0,0,…,-6.814286,-4.909994,-5.376739,-7.487052,-5.326527,-6.483825,-5.432946,-9.442807,-13.571656,-2.074231,-4.420578,-7.045689,-7.602565,-11.253423,-4.028619,-4.881729,-7.984861,-5.044735,-6.333924,-6.462128,-5.457805,-9.109361,-8.434558,-4.340428,-6.308007,-3.500106,-3.557577,-4.335899,-6.04867,-3.293171,-4.357157,-10.874541,-8.247507,-1.350193,-3.740605,-7.833687,-0.436388
1750004,1,0,2,0,2,3,2,2,4,2,0,0,0,1,2,0,2,2,2,0,0,2,2,0,0,0,1,2,1,2,2,0,0,212,0,0,…,-9.349048,-5.171648,-5.341965,-7.851793,-4.644168,-8.172426,-6.484761,-9.873532,-13.317555,-3.120563,-4.404931,-7.442555,-8.799005,-11.117671,-6.310752,-5.769997,-6.847197,-6.182241,-5.586413,-5.510683,-6.574431,-10.58214,-2.812452,-4.797092,-5.467015,-6.521562,-5.397478,-3.660425,-6.213749,-2.886229,-3.730834,-7.399069,-5.814878,-0.486924,-2.036077,-6.79542,-0.633722
1750005,0,1,2,0,2,3,2,2,4,2,0,0,0,1,2,0,2,2,2,1,0,0,2,1,1,0,1,2,1,2,2,1,0,212,0,0,…,-7.971654,-4.688121,-5.452617,-7.410337,-5.793812,-6.067568,-5.885282,-10.019301,-12.301235,-2.264424,-4.759583,-7.120027,-8.689535,-12.003541,-6.707885,-5.882269,-6.376899,-4.427032,-4.90637,-6.94809,-6.309877,-9.398426,-8.146391,-4.637849,-6.604285,-5.710643,-3.241156,-5.179743,-5.612338,-2.624685,-3.618327,-8.062036,-4.580536,-1.560034,-2.087354,-8.707927,-0.612452
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1999996,1,2,2,1,2,3,2,2,4,2,2,2,2,1,2,0,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,2,212,2,2,…,-8.377092,-7.672467,-6.800115,-8.747928,-5.204561,-7.206981,-5.240312,-11.650663,-12.930963,-3.06933,-5.097379,-7.678875,-8.817793,-10.696985,-7.72888,-5.965295,-7.120524,-4.769619,-4.740723,-5.59654,-6.11496,-9.663275,-3.224974,-4.53362,-6.46

## Формирование submission

In [42]:
predict_schema = ["customer_id"]+[col.replace("target_", "predict_") for col in target.columns if col.startswith("target_")]

catboost_predictions = pl.DataFrame(test.select(target.columns), schema = predict_schema)
catboost_predictions

customer_id,predict_1_1,predict_1_2,predict_1_3,predict_1_4,predict_1_5,predict_2_1,predict_2_2,predict_2_3,predict_2_4,predict_2_5,predict_2_6,predict_2_7,predict_2_8,predict_3_1,predict_3_2,predict_3_3,predict_3_4,predict_3_5,predict_4_1,predict_5_1,predict_5_2,predict_6_1,predict_6_2,predict_6_3,predict_6_4,predict_6_5,predict_7_1,predict_7_2,predict_7_3,predict_8_1,predict_8_2,predict_8_3,predict_9_1,predict_9_2,predict_9_3,predict_9_4,predict_9_5,predict_9_6,predict_9_7,predict_9_8,predict_10_1
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1750001,-7.188548,-6.073806,-4.952836,-4.390186,-7.029726,-4.984127,-5.272237,-7.648119,-4.768143,-6.958062,-6.138118,-8.929741,-12.500473,-2.240438,-4.582098,-7.585328,-8.788304,-11.427572,-4.502799,-5.568636,-6.659411,-6.101653,-6.153649,-5.613236,-5.934041,-9.173555,-2.753256,-4.745554,-5.110236,-6.248015,-4.634518,-3.979078,-5.550732,-2.905332,-4.018138,-10.36281,-7.443879,-0.295655,-2.368205,-7.109498,-0.59118
1750002,-5.429091,-5.432858,-3.615,-4.007675,-6.610275,-3.243938,-5.277364,-7.327784,-4.304502,-5.916581,-6.39367,-8.072554,-12.004719,-2.78021,-5.05643,-7.289385,-8.92129,-11.614137,-6.261377,-5.052025,-6.432152,-6.026409,-5.267215,-5.941416,-5.997136,-9.000745,-3.318007,-4.060681,-6.220353,-4.872623,-2.352105,-5.616156,-6.227298,-1.540115,-3.646324,-4.635172,-4.257187,-0.754386,-1.90217,-9.887006,-1.284431
1750003,-5.547932,-5.512655,-3.989439,-3.141071,-6.814286,-4.909994,-5.376739,-7.487052,-5.326527,-6.483825,-5.432946,-9.442807,-13.571656,-2.074231,-4.420578,-7.045689,-7.602565,-11.253423,-4.028619,-4.881729,-7.984861,-5.044735,-6.333924,-6.462128,-5.457805,-9.109361,-8.434558,-4.340428,-6.308007,-3.500106,-3.557577,-4.335899,-6.04867,-3.293171,-4.357157,-10.874541,-8.247507,-1.350193,-3.740605,-7.833687,-0.436388
1750004,-7.566135,-7.603064,-5.278802,-5.239091,-9.349048,-5.171648,-5.341965,-7.851793,-4.644168,-8.172426,-6.484761,-9.873532,-13.317555,-3.120563,-4.404931,-7.442555,-8.799005,-11.117671,-6.310752,-5.769997,-6.847197,-6.182241,-5.586413,-5.510683,-6.574431,-10.58214,-2.812452,-4.797092,-5.467015,-6.521562,-5.397478,-3.660425,-6.213749,-2.886229,-3.730834,-7.399069,-5.814878,-0.486924,-2.036077,-6.79542,-0.633722
1750005,-4.27988,-5.977506,-3.461046,-3.400909,-7.971654,-4.688121,-5.452617,-7.410337,-5.793812,-6.067568,-5.885282,-10.019301,-12.301235,-2.264424,-4.759583,-7.120027,-8.689535,-12.003541,-6.707885,-5.882269,-6.376899,-4.427032,-4.90637,-6.94809,-6.309877,-9.398426,-8.146391,-4.637849,-6.604285,-5.710643,-3.241156,-5.179743,-5.612338,-2.624685,-3.618327,-8.062036,-4.580536,-1.560034,-2.087354,-8.707927,-0.612452
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1999996,-6.051311,-6.874092,-5.995539,-5.042032,-8.377092,-7.672467,-6.800115,-8.747928,-5.204561,-7.206981,-5.240312,-11.650663,-12.930963,-3.06933,-5.097379,-7.678875,-8.817793,-10.696985,-7.72888,-5.965295,-7.120524,-4.769619,-4.740723,-5.59654,-6.11496,-9.663275,-3.224974,-4.53362,-6.4663,-5.602787,-3.564221,-5.912134,-8.503816,-6.413233,-4.249313,-5.525507,-6.638898,-2.108278,-5.594189,-8.196369,1.060799
1999997,-6.529483,-6.389601,-5.011321,-5.018652,-6.680862,-4.547423,-6.606545,-4.735188,-6.332781,-6.847538,-7.423078,-7.826604,-7.426893,-1.534622,-0.759503,-7.413406,-5.877029,-6.830706,-5.102887,-3.827651,-7.40628,-4.365667,-5.011363,-5.322079,-3.856538,-6.294744,-2.710852,-5.312536,-6.660706,-0.822982,-3.792812,-5.766291,-7.093556,-4.048414,-5.45185,-8.275241,-7.298131,-2.78043,-3.284586,-4.113454,-1.826927
1999998,-7.934823,-8.545553,-8.029762,-7.286466,-11.611219,-7.173125,-10.392511,-7.055096,-4.916588,-7.036708,-5.854536,-10.584952,-13.311935,-2.592671,-0.352829,-8.054361,-8.706239,-4.310983,-7.191545,-4.865198,-6.586192,-3.968004,-6.597627,-5.606631,-3.419872,-8.668968,-4.426738,-7.335197,-7.401307,1.902676,-5.559064,-5.067

## Лучшие и худшие предсказанные категории

In [44]:
import glob
import os

results = []
model_files = glob.glob("wocat*.cbm")

for model_file in model_files:
    target_name = model_file.replace("wocat", "").replace(".cbm", "")
    
    model = CatBoostClassifier()
    model.load_model(model_file)
    
    best_iteration = model.get_best_iteration()
    
    best_score = model.get_best_score()
    
    auc_value = None
    if best_score and 'validation' in best_score:
        if 'AUC' in best_score['validation']:
            auc_value = best_score['validation']['AUC']
        else:
            auc_value = list(best_score['validation'].values())[0]
    
    results.append({
        'target_column': target_name,
        'best_iteration': best_iteration,
        'best_AUC': auc_value,
        'model_file': model_file
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('best_AUC', ascending=False).reset_index(drop=True)
results_df['rank'] = results_df.index + 1

print("\n" + "="*80)
print("ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ (из .cbm файлов)")
print("="*80)
print(results_df.to_string(index=False))

results_df.to_csv('ind_clas_41_0818_results.csv', index=False)
print("\nРезультаты сохранены в 'ind_clas_41_0818_results.csv'")

print("\n" + "="*80)
print("РЕЗУЛЬТАТЫ (форматированные)")
print("="*80)
for _, row in results_df.iterrows():
    print(f"{row['target_column']:<20} | "
          f"Итерация: {row['best_iteration']:<6} | "
          f"AUC: {row['best_AUC']:.6f} | "
          f"Ранг: {row['rank']}")


ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ (из .cbm файлов)
target_column  best_iteration  best_AUC           model_file  rank
   target_3_5             948  0.976706  wocattarget_3_5.cbm     1
   target_8_1             998  0.976465  wocattarget_8_1.cbm     2
   target_2_8              25  0.975721  wocattarget_2_8.cbm     3
   target_2_7             362  0.939787  wocattarget_2_7.cbm     4
   target_3_4             778  0.938447  wocattarget_3_4.cbm     5
   target_9_8             989  0.924692  wocattarget_9_8.cbm     6
   target_2_2             941  0.922317  wocattarget_2_2.cbm     7
   target_1_1             808  0.917369  wocattarget_1_1.cbm     8
   target_9_4             287  0.908160  wocattarget_9_4.cbm     9
   target_3_2             911  0.906357  wocattarget_3_2.cbm    10
   target_1_3             999  0.867878  wocattarget_1_3.cbm    11
   target_1_5             799  0.867506  wocattarget_1_5.cbm    12
   target_8_3             915  0.863549  wocattarget_8_3.cbm    13
   target_6_5  

target_column        | best_iteration   | best_AUC      | rank
---------------------|------------------|---------------|----------
target_3_5           | 948    | 0.976706 | 1
target_8_1           | 998    | 0.976465 | 2
target_2_8           | 25     | 0.975721 | 3
target_2_7           | 362    | 0.939787 | 4
target_3_4           | 778    | 0.938447 | 5
target_9_8           | 989    | 0.924692 | 6
target_2_2           | 941    | 0.922317 | 7
target_1_1           | 808    | 0.917369 | 8
target_9_4           | 287    | 0.908160 | 9
target_3_2           | 911    | 0.906357 | 10
target_1_3           | 999    | 0.867878 | 11
target_1_5           | 799    | 0.867506 | 12
target_8_3           | 915    | 0.863549 | 13
target_6_5           | 376    | 0.860899 | 14
target_4_1           | 504    | 0.857183 | 15
target_6_4           | 157    | 0.851358 | 16
target_7_2           | 731    | 0.844874 | 17
target_8_2           | 926    | 0.840782 | 18
target_1_4           | 723    | 0.825604 | 19
target_9_5           | 212    | 0.825221 | 20
target_9_2           | 484    | 0.822147 | 21
target_1_2           | 205    | 0.807214 | 22
target_2_3           | 101    | 0.805713 | 23
target_7_1           | 991    | 0.801374 | 24
target_9_1           | 463    | 0.799278 | 25
target_2_1           | 378    | 0.795340 | 26
target_3_3           | 205    | 0.783547 | 27
target_7_3           | 484    | 0.777418 | 28
target_2_5           | 696    | 0.776152 | 29
target_6_3           | 378    | 0.760624 | 30
target_9_7           | 525    | 0.750255 | 31
target_10_1          | 864    | 0.747157 | 32
target_2_6           | 607    | 0.740591 | 33
target_5_1           | 283    | 0.737932 | 34
target_2_4           | 139    | 0.728115 | 35
target_6_2           | 945    | 0.723372 | 36
target_5_2           | 593    | 0.712871 | 37
target_6_1           | 692    | 0.696081 | 38
target_3_1           | 884    | 0.687275 | 39
target_9_3           | 729    | 0.684947 | 40
target_9_6           | 734    | 0.684200 | 41

Вывод: надо получше поработать над таргетами `target_9_6`, `target_9_3`, `target_3_1`, ... У них ROC-AUC < 0.7, что сильно занижает результативность нашего решения.

Либо попытаться улучшить модель `CatBoost`, либо выбрать для них другую.

Надо посмотреть на эти таргеты под прицелом: вдруг они встречаются в `train` единажды или типо того. То есть мы банально не можем на них обучиться, записей слишком мало (?)